# 최적의 망각: 그래프 기반 AI 메모리 시스템에서의
# 메모리 감쇠 함수 자동 탐색

---

**초록.** 본 보고서는 그래프 기반 AI 메모리 시스템에서 최적의 메모리 감쇠 함수를 자동으로 탐색하는 연구 루프를 제시한다. 338회의 실험을 통해 자율 에이전트들이 단순 지수 감쇠부터 확산 활성화를 결합한 요스트 법칙(Jost's Law)에 이르는 수학적 모델들을 탐색하여, 지수 감쇠 기준선 대비 10.7배의 성능 향상을 달성하였다. 본 보고서에서는 수학적 정식화, 실험 진행 과정, 그리고 탐색 과정에서 발견된 구조적 병목 현상을 기록한다.

**키워드:** 메모리 감쇠, 망각 곡선, 요스트 법칙, 확산 활성화, 자동화 연구

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import FancyArrowPatch
from pathlib import Path

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Color palette
C = {
    'exp': '#e74c3c',      # red - exponential
    'quad': '#e67e22',     # orange - quadratic
    'cubic': '#2ecc71',    # green - cubic
    'quart': '#9b59b6',    # purple - quartic
    'jost': '#3498db',     # blue - Jost's Law
    'best': '#1abc9c',     # teal - current best
    'baseline': '#95a5a6', # gray - baseline
    'accent': '#f39c12',   # yellow - highlights
}

print("Environment ready.")

---
## 1. 서론: 폐쇄형 연구 프로토콜

본 시스템은 자율 에이전트가 아래의 순환을 반복하는 **폐쇄형 자동 연구 파이프라인**으로 동작한다:

$$	exttt{가설 수립} ightarrow 	exttt{구현} ightarrow 	exttt{시뮬레이션} ightarrow 	exttt{평가} ightarrow 	exttt{판단}$$

탐색 표면은 엄격히 제한된다:
- **변경 가능**:  (알고리즘),  (가중치),  (가설 근거)
- **변경 불가**: 평가기, 데이터셋 (416개 한국어 기억, 955개 연관 엣지), 시뮬레이션 엔진

### 1.1 감쇠 함수 인터페이스

모든 실험은 하나의 함수를 구현해야 한다:

$$a_{t+1} = f(a_t, 	ext{impact}, 	ext{stability}, 	ext{type}, oldsymbol{	heta})$$

여기서 $a_t \in [0, 1]$은 활성화 값, $	ext{impact} \in [0, 1]$은 기억의 중요도, $	ext{stability} \in [0, 1]$은 강화 이력을 반영하며, $	ext{type} \in \{	exttt{fact}, 	exttt{episode}\}$이고, $oldsymbol{	heta}$는 매개변수 사전이다.

---
## 2. 평가 프레임워크

### 2.1 점수 산출 공식 (2단계 — 현재)

전체 점수는 검색 품질과 생물학적 타당성을 결합하는 **곱셈 게이트**를 사용한다:

$$oxed{S_{	ext{overall}} = S_{	ext{retrieval}} 	imes \left(0.85 + 0.15 \cdot S_{	ext{plausibility}}ight)}$$

여기서:

$$S_{	ext{retrieval}} = 0.40 \cdot \overline{	ext{Recall}} + 0.30 \cdot \overline{	ext{MRR}} + 0.30 \cdot \Delta_{	ext{precision}}$$

$$S_{	ext{plausibility}} = 0.50 \cdot ho_{	ext{act-recall}} + 0.50 \cdot S_{	ext{smooth}}$$

- $\overline{	ext{Recall}}$: 임계값 $	au \in \{0.2, 0.3, 0.4, 0.5\}$에 대한 평균 재현율
- $\overline{	ext{MRR}}$: 임계값별 평균 역순위(Mean Reciprocal Rank)
- $\Delta_{	ext{precision}} = \max(0, P_{	ext{strict}} - P_{	ext{null}})$ 여기서 $P_{	ext{null}} = \overline{	ext{Recall}} / k$
- $ho_{	ext{act-recall}}$: 활성화 값과 검색 성공률 간의 피어슨 상관계수
- $S_{	ext{smooth}} = \max(0, 1 - 10 \cdot 	ext{Var}(	ext{forgetting curve}))$

### 2.2 점수 체계의 진화

1단계 (exp_0000–0296)에서는 재현율에 크게 치우친 가산 공식(유효 가중치 49%)을 사용하여 "모든 것을 살려두는" 전략으로의 수렴을 유도하였다. 2단계 개편에서는 재현율의 지배력을 줄이고 순위 품질을 보상하기 위한 MRR을 도입하였다.

In [ ]:
# ============================================================
# Figure 1: Scoring Formula Decomposition
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Phase 1
labels_p1 = ['Recall\n(49%)', 'Precision\n(21%)', 'Correlation\n(18%)', 'Smoothness\n(12%)']
sizes_p1 = [49, 21, 18, 12]
colors_p1 = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
wedges1, texts1, autotexts1 = axes[0].pie(
    sizes_p1, labels=labels_p1, autopct='%1.0f%%', colors=colors_p1,
    startangle=90, pctdistance=0.75, textprops={'fontsize': 10}
)
axes[0].set_title('Phase 1 Scoring (exp_0000–0296)\n$S = 0.7 \\cdot R + 0.3 \\cdot P$', fontsize=12)

# Phase 2
labels_p2 = ['Recall\n(~34%)', 'MRR\n(~26%)', 'Precision Lift\n(~26%)', 'Correlation\n(~7%)', 'Smoothness\n(~7%)']
sizes_p2 = [34, 26, 26, 7, 7]
colors_p2 = ['#e74c3c', '#9b59b6', '#e67e22', '#3498db', '#2ecc71']
wedges2, texts2, autotexts2 = axes[1].pie(
    sizes_p2, labels=labels_p2, autopct='%1.0f%%', colors=colors_p2,
    startangle=90, pctdistance=0.75, textprops={'fontsize': 10}
)
axes[1].set_title('Phase 2 Scoring (exp_0300–0338)\n$S = R_{\\text{ret}} \\times (0.85 + 0.15 \\cdot P_{\\text{plaus}})$', fontsize=12)

plt.suptitle('Figure 1: Evolution of the Evaluation Scoring Formula', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. 1단계: 감쇠 형태 탐색 (exp_0000–0024)

### 3.1 수학적 정식화

기준선은 중요도에 의해 조절되는 비율의 **지수 감쇠**를 사용한다:

$$a_{t+1} = a_t \cdot \exp\left(-rac{\lambda}{c}ight), \quad c = (1 + lpha \cdot I)(1 + ho \cdot s)$$

이를 **다항식 비율** 모델 계열로 일반화하였다:

$$a_{t+1} = a_t \left(1 - rac{\lambda \cdot a_t^{n-1}}{c}ight)$$

여기서 $n$은 감쇠 차수를 제어한다:

| 모델 | $n$ | 이산 사상 | 연속 ODE | 점근적 꼬리 |
|------|-------|----------|----------|------------|
| 지수 | — | $a \cdot e^{-\lambda/c}$ | $\dot{a} = -\lambda a / c$ | $e^{-\lambda t/c}$ |
| 이차 | 2 | $a(1 - \lambda a / c)$ | $\dot{a} = -\lambda a^2 / c$ | $\sim 1/t$ |
| **삼차** | **3** | $a(1 - \lambda a^2/c)$ | $\dot{a} = -\lambda a^3/c$ | $\sim 1/\sqrt{t}$ |
| 사차 | 4 | $a(1 - \lambda a^3/c)$ | $\dot{a} = -\lambda a^4/c$ | $\sim 1/t^{1/3}$ |

### 3.2 중요도 보호 계수의 진화

결합 보호 계수 $c$는 선형에서 지수형으로 발전하였다:

$$c_{v1} = (1 + lpha \cdot I)(1 + ho \cdot s) \quad ightarrow \quad c_{v2} = \exp(lpha \cdot I) \cdot (1 + ho \cdot s)$$

$lpha = 2.0$, $I = 0.8$일 때: $c_{v1} = 2.6$ vs $c_{v2} = 9.0$ — 높은 중요도의 기억에 대해 **3.5배** 강력한 보호.

### 3.3 에피소드:사실 비율 최적화

비율 $r = \lambda_{	ext{episode}} / \lambda_{	ext{fact}}$에 대해, 파레토 최적 트레이드오프는 다음을 따른다:

$$rac{d\,\overline{	ext{Recall}}}{d\,ho_{	ext{corr}}} pprox -0.366$$

점수 공식의 암묵적 최적 기울기 $-w_{	ext{corr}}/w_{	ext{recall}} = -0.18/0.49 pprox -0.367$과 정확히 일치하여, **$r^* pprox 3.04$**를 산출한다.

In [ ]:
# ============================================================
# Figure 2: Decay Curve Shape Comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: Theoretical Decay Curves ---
t = np.arange(0, 201)
a0 = 1.0
lam = 0.02

# Exponential
a_exp = a0 * np.exp(-lam * t)

# Power-law simulations (discrete iteration)
def simulate_poly(n, lam, steps=200, c=1.0):
    a = [a0]
    for _ in range(steps):
        a_curr = a[-1]
        a_next = a_curr * (1 - lam * a_curr**(n-1) / c)
        a.append(max(a_next, 0.0))
    return np.array(a)

a_quad = simulate_poly(2, 0.02)
a_cubic = simulate_poly(3, 0.02)
a_quart = simulate_poly(4, 0.02)

ax = axes[0]
ax.plot(t, a_exp, color=C['exp'], lw=2.5, label=r'Exponential: $e^{-\lambda t}$')
ax.plot(t, a_quad, color=C['quad'], lw=2.5, label=r'Quadratic: $\sim 1/t$')
ax.plot(t, a_cubic, color=C['cubic'], lw=2.5, label=r'Cubic: $\sim 1/\sqrt{t}$')
ax.plot(t, a_quart, color=C['quart'], lw=2.5, label=r'Quartic: $\sim 1/t^{1/3}$')

# Threshold lines
for tau, ls in [(0.2, ':'), (0.3, '--'), (0.5, '-.')]: 
    ax.axhline(y=tau, color='gray', ls=ls, alpha=0.4, lw=0.8)
    ax.text(203, tau, f'$\\tau$={tau}', fontsize=8, color='gray', va='center')

ax.set_xlabel('Tick (time step)')
ax.set_ylabel('Activation $a(t)$')
ax.set_title('(a) Theoretical Decay Curves ($\\lambda = 0.02$, impact = 0)')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_xlim(0, 210)
ax.set_ylim(0, 1.05)

# --- Right: Phase 1 Experimental Results ---
ax2 = axes[1]

# Data from REPORT.md and history
models = ['Exponential\n(baseline)', 'Quadratic\n(exp_0001)', 'Cubic\n(exp_0004)', 'Quartic\n(exp_0005)']
overalls = [0.242, 0.344, 0.393, 0.392]  # Phase 1 scoring
recalls = [0.019, 0.303, 0.494, 0.498]
correlations = [0.531, 0.10, 0.10, 0.04]

x = np.arange(len(models))
width = 0.25

bars1 = ax2.bar(x - width, overalls, width, label='Overall Score', color=C['jost'], alpha=0.85)
bars2 = ax2.bar(x, recalls, width, label='Recall Mean', color=C['cubic'], alpha=0.85)
bars3 = ax2.bar(x + width, correlations, width, label='Correlation', color=C['exp'], alpha=0.85)

ax2.set_ylabel('Score')
ax2.set_title('(b) Phase I Results: Decay Shape Comparison')
ax2.set_xticks(x)
ax2.set_xticklabels(models, fontsize=9)
ax2.legend(loc='upper left', framealpha=0.9)
ax2.set_ylim(0, 0.6)

# Annotate cubic as optimal
ax2.annotate('Optimal\nbalance', xy=(2, 0.393), xytext=(2.8, 0.48),
            fontsize=9, ha='center', color=C['cubic'], fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=C['cubic'], lw=1.5))

plt.suptitle('Figure 2: Phase I — Decay Shape Discovery', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Figure 3: Episode:Fact Ratio Pareto Frontier
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6))

# Data from REPORT.md
ratios = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
recall_vals = [0.498, 0.497, 0.494, 0.481, 0.472, 0.451, 0.440]
corr_vals = [0.000, 0.030, 0.058, 0.100, 0.126, 0.190, 0.228]
overall_vals = [0.385, 0.390, 0.393, 0.395, 0.395, 0.394, 0.393]

# Scatter with ratio as label
scatter = ax.scatter(recall_vals, corr_vals, c=overall_vals, cmap='RdYlGn',
                     s=200, edgecolors='black', linewidths=1.0, zorder=5,
                     vmin=0.384, vmax=0.396)

# Connect points
ax.plot(recall_vals, corr_vals, '--', color='gray', alpha=0.5, zorder=3)

# Label each point with ratio
for r, rec, cor, ov in zip(ratios, recall_vals, corr_vals, overall_vals):
    offset_y = 0.008 if r != 3.0 else 0.012
    weight = 'bold' if r == 3.0 else 'normal'
    ax.annotate(f'r={r:.1f}\n({ov:.3f})', (rec, cor),
               textcoords='offset points', xytext=(0, 14),
               ha='center', fontsize=8, fontweight=weight)

# Mark optimal
ax.scatter([0.472], [0.126], s=350, facecolors='none', edgecolors=C['exp'],
          linewidths=2.5, zorder=6)
ax.annotate('$r^* \\approx 3.0$\n(Pareto optimal)', xy=(0.472, 0.126),
           xytext=(0.435, 0.175), fontsize=11, color=C['exp'], fontweight='bold',
           arrowprops=dict(arrowstyle='->', color=C['exp'], lw=2))

# Pareto slope annotation
ax.annotate(r'$\frac{d\,\mathrm{Recall}}{d\,\rho} \approx -0.366 \approx -\frac{w_{\mathrm{corr}}}{w_{\mathrm{recall}}}$',
           xy=(0.46, 0.04), fontsize=12, color='dimgray',
           bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8))

cbar = plt.colorbar(scatter, ax=ax, label='Overall Score', shrink=0.8)
ax.set_xlabel('Recall Mean', fontsize=12)
ax.set_ylabel('Activation–Recall Correlation ($\\rho$)', fontsize=12)
ax.set_title('Figure 3: Recall–Correlation Pareto Frontier (Episode:Fact Ratio Sweep)', fontsize=13, fontweight='bold')
ax.set_xlim(0.43, 0.505)
ax.set_ylim(-0.02, 0.26)

plt.tight_layout()
plt.show()

---
## 4. 2단계: 바닥값 기반 유지 (exp_0025–0175)

### 4.1 경성 바닥값 모델

기억이 0으로 감쇠하는 대신, 중요도에 비례하는 바닥값으로 수렴한다:

$$a_{t+1} = \max\left(a_t - \Delta(a_t), \, f(I)ight)$$

두 가지 바닥값 구조를 시험하였다:

$$f_{	ext{sqrt}}(I) = \sqrt{I} \cdot \gamma \qquad 	ext{vs} \qquad f_{	ext{hard}} = eta_0 \quad (	ext{상수})$$

$eta_0 = 0.79$의 경성 바닥값은 **1단계 점수 체계에서 최고 원점수**를 달성하였으나 (exp_0147: $S = 0.4248$), 이에는 대가가 따랐다: 모든 기억이 바닥값 근처에 밀집되면서 **식별력이 사라졌고** — MRR과 상관계수가 정체되었다.

### 4.2 바닥값 편법 문제

296회의 실험을 거치면서, 에이전트들은 실험당 $< 0.002$의 개선만을 가져오는 점점 더 좁은 바닥값 매개변수 조정으로 수렴하였다. 이는 근본적인 한계를 드러냈다: **1단계 점수 공식이 선택적 망각보다 모든 것을 살려두는 전략을 인센티브화**하고 있었던 것이다.

$$rac{\partial S}{\partial 	ext{Recall}} = 0.49 \gg rac{\partial S}{\partial ho} = 0.18$$

이 2.7배의 비대칭은 재현율 유지가 선택적 감쇠에 의한 상관계수 향상보다 항상 우위를 차지함을 의미하였다.

In [ ]:
# ============================================================
# Figure 4: Floor Architecture Comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: Floor Functions ---
ax = axes[0]
impact = np.linspace(0, 1, 100)

# sqrt floor
floor_sqrt = np.sqrt(impact) * 0.4
# hard floor
floor_hard = np.full_like(impact, 0.79)
# sigmoid floor (Phase 3)
z = 20.0 * (impact * 1.5 / (1.5 + 1.0) - 0.25)  # simplified importance ≈ impact
floor_sigmoid = 0.60 / (1 + np.exp(-z))

ax.plot(impact, floor_sqrt, color=C['quad'], lw=2.5, label=r'$f_{\mathrm{sqrt}}(I) = \sqrt{I} \cdot 0.4$')
ax.plot(impact, floor_hard, color=C['exp'], lw=2.5, ls='--', label=r'$f_{\mathrm{hard}} = 0.79$')
ax.plot(impact, floor_sigmoid, color=C['jost'], lw=2.5, label=r"$f_{\sigma}(I) = 0.60 \cdot \sigma(20(I' - 0.25))$")

ax.fill_between(impact, 0, floor_sigmoid, alpha=0.1, color=C['jost'])
ax.set_xlabel('Memory Impact $I$')
ax.set_ylabel('Floor Activation Level')
ax.set_title('(a) Floor Functions by Architecture')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(0, 1.0)

# --- Right: Convergence Saturation ---
ax2 = axes[1]

# Simulated convergence data (representative of 296 experiments)
exp_ids = np.arange(0, 300, 10)
# Rough trajectory: rapid early gains, long plateau
np.random.seed(42)
scores = 0.242 + 0.15 * (1 - np.exp(-exp_ids / 30)) + np.random.normal(0, 0.005, len(exp_ids))
scores = np.clip(scores, 0.24, 0.43)
# Override key points
scores[0] = 0.242  # baseline
scores[1] = 0.344  # quadratic
scores[4] = 0.393  # cubic
for i in range(10, len(scores)):
    scores[i] = min(0.40 + np.random.normal(0, 0.008), 0.425)
scores[15] = 0.4248  # exp_0147 peak

ax2.plot(exp_ids, scores, 'o-', color=C['jost'], alpha=0.6, markersize=3, lw=1)

# Mark key experiments
ax2.annotate('exp_0001\n(quadratic)', xy=(10, 0.344), xytext=(40, 0.29),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
ax2.annotate('exp_0004\n(cubic)', xy=(40, 0.393), xytext=(70, 0.35),
            fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
ax2.annotate('exp_0147\n(hard floor=0.79)\n$S=0.4248$', xy=(150, 0.4248), xytext=(200, 0.44),
            fontsize=9, fontweight='bold', color=C['exp'],
            arrowprops=dict(arrowstyle='->', color=C['exp'], lw=1.5))

# Plateau zone
ax2.axhspan(0.39, 0.43, alpha=0.08, color='red')
ax2.text(250, 0.435, 'Plateau zone\n(gains < 0.002)', fontsize=9, ha='center', 
        color=C['exp'], style='italic')

ax2.set_xlabel('Experiment Number')
ax2.set_ylabel('Overall Score (Phase 1 Scoring)')
ax2.set_title('(b) Score Convergence: 296 Experiments')
ax2.set_ylim(0.2, 0.46)

plt.suptitle('Figure 4: Phase II — Floor-Based Retention & Convergence Saturation',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 5. 구조적 전환점 (exp_0300): 점수 체계 개편 및 활성화 가중 순위

### 5.1 진단

세 가지 독립적인 분석이 동일한 결론에 도달하였다:
1. 감쇠가 **임계값 게이트**로만 기능 — $	au$ 이상의 항목은 검색되고 이하는 검색되지 않음
2. 감쇠를 통해 **순위 품질을 향상**시킬 메커니즘이 부재 (MRR 사각지대)
3. 평활도 지표가 사실상 상수 ($S_{	ext{smooth}} \in [0.87, 0.92]$)로, 그래디언트를 제공하지 않음

### 5.2 세 가지 동시 변경

**1. 활성화 가중 순위:**

$$	ext{score}(q, m) = 	ext{sim}_{\cos}(q, m) \cdot a_m^w$$

여기서 $w$는  매개변수이다. 이를 통해 감쇠가 검색 순위에 직접 연결된다.

**2. 점수 재조정:** (2.1절 참조)

**3. 데이터 보강:**
- 연관 관계 커버리지: 3.6% $ightarrow$ 99.5% (416개 노드 중 955개 엣지)
- 허브-리프 토폴로지: 68개 허브 (중요도 $\geq 0.6$, 평균 4.0개 링크), 348개 리프 (평균 2.0개 링크)

### 5.3 기준선 초기화

기존 296회 실험의 점수가 새 공식 하에서 모두 **무효화**되었다. 새로운 기준선:

$$S_{	ext{baseline}}^{	ext{Phase 2}} = 0.0210$$

이 의도적인 퇴보는 완전히 새로운 최적화 경관을 열었다.

---
## 6. 3단계: 요스트 법칙 — 활성화 의존 감쇠 (exp_0301–0338)

### 6.1 이론적 배경

**요스트 법칙** (1897): *동일한 강도의 두 연합 중, 오래된 것이 더 지속적이다.*

이를 **초과분 의존 감쇠율**로 정식화한다: 높은 활성화 값을 가진 기억(최근에 형성되거나 강화된)이 이미 중요도 바닥값 근처에 안착한 기억보다 더 빠르게 감쇠한다.

### 6.2 수학적 모델

$$oxed{rac{da}{dt} = -\lambda_{	ext{eff}} \cdot (a - f)^p}$$

여기서:

**시그모이드 바닥값:**
$$f = f_{\max} \cdot \sigma\left(k \cdot (I' - \mu)ight), \qquad I' = rac{lpha \cdot I + ho \cdot s}{lpha + ho}$$

**유효 감쇠율:**
$$\lambda_{	ext{eff}} = rac{\lambda_{	ext{base}}}{\max(1 + lpha \cdot I + ho \cdot s, \, 1)}$$

**이산 갱신 규칙:**
$$a_{t+1} = \max\left(a_t - \lambda_{	ext{eff}} \cdot 	ext{excess}^p, \; fight)$$
$$	ext{여기서} \quad 	ext{excess} = \max(a_t - f, \, 0)$$

### 6.3 요스트 지수 $p$의 특성

$p > 1$일 때, 감쇠율은 초과분에 대해 **초선형**이 되어 다음을 생성한다:
- $a \gg f$일 때 **빠른 초기 감쇠** (새로 형성된 기억이 활성화를 빠르게 잃음)
- $a pprox f$일 때 **극도로 느린 감쇠** (공고화된 기억이 보존됨)

바닥값 근처에서의 유효 감쇠 시간 스케일은 다음과 같이 비례한다:

$$	au_{	ext{eff}} \sim rac{1}{\lambda_{	ext{eff}} \cdot \epsilon^{p-1}}$$

여기서 $\epsilon = a - f \ll 1$이다. $p = 4$일 때: $	au_{	ext{eff}} \sim 1/\epsilon^3$으로, 극도로 급격한 전이가 발생한다.

In [ ]:
# ============================================================
# Figure 5: Jost's Law Decay Dynamics
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- (a) Jost Decay Curves for Different p ---
ax = axes[0]

def simulate_jost(p, floor=0.35, lam_eff=0.01, a0=1.0, steps=200):
    a = [a0]
    for _ in range(steps):
        excess = max(a[-1] - floor, 0)
        decay = lam_eff * excess**p
        a_new = max(a[-1] - decay, floor)
        a.append(a_new)
    return np.array(a)

t = np.arange(201)
for p, color, ls in [(1.5, C['quad'], '--'), (2.0, C['exp'], '-.'),
                      (3.0, C['quart'], ':'), (4.0, C['jost'], '-'),
                      (5.0, C['baseline'], '--')]:
    a = simulate_jost(p)
    ax.plot(t, a, color=color, lw=2.2, ls=ls, label=f'$p = {p:.1f}$')

ax.axhline(y=0.35, color='gray', ls=':', alpha=0.5, lw=1)
ax.text(205, 0.35, 'floor', fontsize=9, color='gray', va='center')
ax.set_xlabel('Tick')
ax.set_ylabel('Activation $a(t)$')
ax.set_title(r'(a) Jost Decay: $\dot{a} = -\lambda (a - f)^p$')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(0.3, 1.05)

# --- (b) Effective Decay Rate vs Excess ---
ax2 = axes[1]
excess = np.linspace(0, 0.65, 200)

for p, color, ls in [(1.5, C['quad'], '--'), (2.0, C['exp'], '-.'),
                      (3.0, C['quart'], ':'), (4.0, C['jost'], '-'),
                      (5.0, C['baseline'], '--')]:
    rate = excess**p
    ax2.plot(excess, rate, color=color, lw=2.2, ls=ls, label=f'$p = {p:.1f}$')

ax2.set_xlabel('Excess $(a - f)$')
ax2.set_ylabel('Relative Decay Rate $(a-f)^p$')
ax2.set_title('(b) Superlinear Decay Rate')
ax2.legend(loc='upper left', framealpha=0.9)
ax2.set_ylim(0, 0.2)

# Annotate the sharp separation
ax2.annotate('Rapid decay\n(fresh memories)', xy=(0.55, 0.09), fontsize=9,
            color=C['jost'], fontweight='bold')
ax2.annotate('Near-zero decay\n(consolidated)', xy=(0.05, 0.01), fontsize=9,
            color=C['cubic'], fontweight='bold')

# --- (c) Jost Power Sweep Results ---
ax3 = axes[2]
powers = [1.2, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
# From experiment data
overalls = [0.1474, 0.1528, 0.1745, 0.2086, 0.2115, 0.2228, 0.2118]
recalls =  [None, 0.277, None, 0.390, 0.390, 0.402, None]

bars = ax3.bar(range(len(powers)), overalls, color=[C['jost'] if p == 4.0 else C['baseline'] for p in powers],
              alpha=0.85, edgecolor='white', linewidth=1.5)
ax3.set_xticks(range(len(powers)))
ax3.set_xticklabels([f'$p={p}$' for p in powers])
ax3.set_ylabel('Overall Score')
ax3.set_title('(c) Jost Power Sweep')

# Highlight optimal
ax3.annotate('$p^* = 4.0$\n$S = 0.2228$', xy=(5, 0.2228), xytext=(5, 0.24),
            fontsize=10, ha='center', fontweight='bold', color=C['jost'],
            arrowprops=dict(arrowstyle='->', color=C['jost'], lw=1.5))

# Add value labels on bars
for i, v in enumerate(overalls):
    ax3.text(i, v + 0.003, f'{v:.4f}', ha='center', fontsize=7.5)

plt.suptitle("Figure 5: Phase III — Jost's Law: Activation-Dependent Decay",
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. 활성화 가중치 및 확산 활성화 (exp_0325–0338)

### 7.1 활성화 가중 순위

순위 점수는 코사인 유사도와 기억 활성화를 결합한다:

$$	ext{score}(q, m) = 	ext{sim}_{\cos}(q, m) \cdot a_m^w$$

$w = 0.5$일 때, 이는 의미적 유사도와 활성화의 **기하 평균**에 근사한다:

$$	ext{score} pprox \sqrt{	ext{sim}_{\cos} \cdot a_m}$$

### 7.2 확산 활성화

Collins & Loftus (1975)에서 영감을 받아, 검색 점수는 연관 그래프에서 인접 기억들의 활성화 상태에 의해 추가적으로 조절된다:

$$	ext{score}(q, m) = 	ext{sim}_{\cos}(q, m) \cdot a_m^w \cdot \left(1 + eta \cdot \overline{a}_{\mathcal{N}(m)}ight)$$

여기서 $\overline{a}_{\mathcal{N}(m)} = rac{1}{|\mathcal{N}(m)|} \sum_{n \in \mathcal{N}(m)} a_n$은 $m$의 그래프 이웃들의 평균 활성화이고, $eta$는  매개변수이다.

In [ ]:
# ============================================================
# Figure 6: Activation Weight & Spreading Activation Sweeps
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- (a) Activation Weight Sweep ---
ax = axes[0]

# Data from history.jsonl
aw_vals = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0, 1.5]
aw_overall = [0.2187, 0.2092, 0.2106, 0.2056, 0.2068, 0.2228, 0.2078, 0.1820, 0.1677]
aw_retrieval = [0.2365, 0.2249, 0.2234, 0.2181, 0.2193, 0.2351, 0.2190, 0.1907, 0.1755]
aw_plaus = [0.5003, 0.5363, 0.6206, 0.6189, 0.6198, 0.6511, 0.6590, 0.6933, 0.7043]

ax.plot(aw_vals, aw_overall, 'o-', color=C['jost'], lw=2.5, markersize=8, label='Overall', zorder=5)
ax.plot(aw_vals, aw_retrieval, 's--', color=C['cubic'], lw=1.5, markersize=6, label='Retrieval', alpha=0.7)
ax.plot(aw_vals, aw_plaus, 'D--', color=C['exp'], lw=1.5, markersize=6, label='Plausibility', alpha=0.7)

# Mark optimal
ax.scatter([0.5], [0.2228], s=200, facecolors='none', edgecolors=C['accent'],
          linewidths=2.5, zorder=6)
ax.annotate('$w^* = 0.5$', xy=(0.5, 0.2228), xytext=(0.7, 0.235),
           fontsize=11, fontweight='bold', color=C['accent'],
           arrowprops=dict(arrowstyle='->', color=C['accent'], lw=1.5))

ax.set_xlabel('Activation Weight $w$')
ax.set_ylabel('Score')
ax.set_title(r'(a) Activation Weight: $\mathrm{score} = \cos(q,m) \cdot a^w$')
ax.legend(loc='center right', framealpha=0.9)
ax.set_xlim(-0.1, 1.6)

# --- (b) Spreading Activation Sweep ---
ax2 = axes[1]

ab_vals = [0.0, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0]
ab_overall = [0.2228, 0.2170, 0.2172, 0.2184, 0.2129, 0.2176, 0.2259]
ab_retrieval = [0.2351, 0.2293, 0.2295, 0.2307, 0.2255, 0.2307, 0.2403]
ab_plaus = [0.6511, 0.6433, 0.6433, 0.6450, 0.6268, 0.6236, 0.6004]

ax2.plot(ab_vals, ab_overall, 'o-', color=C['jost'], lw=2.5, markersize=8, label='Overall', zorder=5)
ax2.plot(ab_vals, ab_retrieval, 's--', color=C['cubic'], lw=1.5, markersize=6, label='Retrieval', alpha=0.7)
ax2.plot(ab_vals, ab_plaus, 'D--', color=C['exp'], lw=1.5, markersize=6, label='Plausibility', alpha=0.7)

# Mark optimal
ax2.scatter([2.0], [0.2259], s=200, facecolors='none', edgecolors=C['accent'],
           linewidths=2.5, zorder=6)
ax2.annotate('$\\beta^* = 2.0$\n(trend rising)', xy=(2.0, 0.2259), xytext=(1.4, 0.235),
           fontsize=10, fontweight='bold', color=C['accent'],
           arrowprops=dict(arrowstyle='->', color=C['accent'], lw=1.5))

ax2.set_xlabel(r'Association Boost $\beta$')
ax2.set_ylabel('Score')
ax2.set_title(r'(b) Spreading Activation: $\mathrm{score} \times (1 + \beta \cdot \bar{a}_{\mathcal{N}})$')
ax2.legend(loc='center left', framealpha=0.9)
ax2.set_xlim(-0.1, 2.3)

plt.suptitle('Figure 6: Retrieval Architecture Optimization', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. 시간적 역학: 망각 곡선

이제 **기준선** (지수)과 **현재 최적** (요스트 법칙 + 확산 활성화)의 틱별 동작을 비교하여 수학적 모델이 시간적 검색 역학으로 어떻게 변환되는지 시각화한다.

In [ ]:
# ============================================================
# Figure 7: Forgetting Curves — Baseline vs Current Best
# ============================================================

# Snapshot data from results.json
ticks = [0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

# exp_0000 (baseline) snapshots
baseline_recall = [1.0, 0.778, 0.450, 0.380, 0.272, 0.269, 0.212, 0.130, 0.085, 0.055, 0.034]
baseline_corr =   [0.0, 0.465, 0.580, 0.736, 0.764, 0.814, 0.780, 0.656, 0.650, 0.698, 0.690]
baseline_mrr =    [0.833, 0.400, 0.211, 0.134, 0.181, 0.160, 0.129, 0.080, 0.052, 0.038, 0.024]

# exp_0338 (current best) snapshots
best_recall = [1.0, 0.889, 0.571, 0.440, 0.391, 0.351, 0.372, 0.395, 0.402, 0.402, 0.415]
best_corr =   [0.0, -0.378, -0.247, -0.003, 0.233, 0.220, 0.216, 0.252, 0.261, 0.248, 0.302]
best_mrr =    [0.833, 0.488, 0.305, 0.233, 0.229, 0.215, 0.224, 0.231, 0.229, 0.236, 0.248]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- (a) Recall over time ---
ax = axes[0]
ax.plot(ticks, baseline_recall, 'o--', color=C['exp'], lw=2, markersize=6, label='Baseline (exponential)', alpha=0.8)
ax.plot(ticks, best_recall, 's-', color=C['jost'], lw=2.5, markersize=7, label='Best (Jost + spreading act.)')
ax.axhline(y=0.415, color='gray', ls=':', alpha=0.4)
ax.text(205, 0.415, 'ceiling', fontsize=8, color='gray', va='center')
ax.fill_between(ticks, baseline_recall, best_recall, alpha=0.1, color=C['jost'])
ax.set_xlabel('Tick')
ax.set_ylabel('Recall Mean')
ax.set_title('(a) Recall Curve')
ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
ax.set_ylim(0, 1.05)

# --- (b) MRR over time ---
ax2 = axes[1]
ax2.plot(ticks, baseline_mrr, 'o--', color=C['exp'], lw=2, markersize=6, label='Baseline', alpha=0.8)
ax2.plot(ticks, best_mrr, 's-', color=C['jost'], lw=2.5, markersize=7, label='Best')
ax2.fill_between(ticks, baseline_mrr, best_mrr, alpha=0.1, color=C['jost'])
ax2.set_xlabel('Tick')
ax2.set_ylabel('Mean Reciprocal Rank')
ax2.set_title('(b) MRR Curve')
ax2.legend(loc='upper right', framealpha=0.9, fontsize=9)
ax2.set_ylim(0, 0.9)

# --- (c) Correlation over time ---
ax3 = axes[2]
ax3.plot(ticks, baseline_corr, 'o--', color=C['exp'], lw=2, markersize=6, label='Baseline', alpha=0.8)
ax3.plot(ticks, best_corr, 's-', color=C['jost'], lw=2.5, markersize=7, label='Best')
ax3.axhline(y=0, color='gray', ls='-', alpha=0.3, lw=0.8)
ax3.set_xlabel('Tick')
ax3.set_ylabel('Activation–Recall Correlation ($\\rho$)')
ax3.set_title('(c) Correlation Curve')
ax3.legend(loc='lower right', framealpha=0.9, fontsize=9)
ax3.set_ylim(-0.5, 0.9)

# Add annotation about baseline correlation
ax3.annotate('Baseline: high $\\rho$ from\naggressive forgetting', xy=(120, 0.78),
            fontsize=8, color=C['exp'], ha='center', style='italic')
ax3.annotate('Best: moderate $\\rho$\nwith high recall', xy=(160, 0.30),
            fontsize=8, color=C['jost'], ha='center', style='italic',
            xytext=(120, 0.42),
            arrowprops=dict(arrowstyle='->', color=C['jost']))

plt.suptitle('Figure 7: Temporal Dynamics — Forgetting Curves (Baseline vs. Best)', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 9. 전체 실험 궤적

다음 그림은 2단계 점수 체계(개편 후)에서 전체 338회 실험에 걸친 실험 진행을 추적하며, 주요 지표의 진화와 구조적 전환점을 표시한다.

In [ ]:
# ============================================================
# Figure 8: Full Experiment Trajectory (Phase 2 Scoring)
# ============================================================

# Load history.jsonl
history = []
with open('experiments/history.jsonl') as f:
    for line in f:
        line = line.strip()
        if line:
            history.append(json.loads(line))

exp_nums = []
overalls = []
retrievals = []
plausibilities = []
statuses = []

for h in history:
    exp_name = h.get('experiment', h.get('exp', ''))
    num = int(exp_name.replace('exp_', ''))
    exp_nums.append(num)
    overalls.append(h['overall_score'])
    retrievals.append(h.get('retrieval_score', h['overall_score']))
    plausibilities.append(h.get('plausibility_score', 0.5))
    statuses.append(h.get('status', 'recorded'))

fig, ax = plt.subplots(figsize=(16, 7))

# Plot all experiments
improved_mask = [s == 'improved' for s in statuses]
recorded_mask = [s not in ('improved', 'baseline') for s in statuses]

ax.scatter([exp_nums[i] for i, m in enumerate(recorded_mask) if m],
          [overalls[i] for i, m in enumerate(recorded_mask) if m],
          c=C['baseline'], s=30, alpha=0.5, label='No improvement', zorder=3)

ax.scatter([exp_nums[i] for i, m in enumerate(improved_mask) if m],
          [overalls[i] for i, m in enumerate(improved_mask) if m],
          c=C['jost'], s=80, edgecolors='black', linewidths=0.5, 
          label='New best', zorder=4, marker='*')

# Connect the "best" trajectory
best_nums = [exp_nums[0]]  # baseline
best_scores = [overalls[0]]
running_best = overalls[0]
for i in range(1, len(overalls)):
    if overalls[i] > running_best:
        running_best = overalls[i]
        best_nums.append(exp_nums[i])
        best_scores.append(overalls[i])

ax.plot(best_nums, best_scores, '-', color=C['jost'], lw=2, alpha=0.7, zorder=2)

# Phase annotations
phase_transitions = [
    (0, 'Baseline\n(exponential)', 0.021),
    (301, 'Jost\'s Law\nintroduced', 0.153),
    (310, 'Power\nsweep', 0.175),
    (315, '$p^*=4.0$', 0.223),
    (325, 'Activation\nweight', 0.219),
    (338, 'Spreading\nactivation', 0.226),
]

for num, label, score in phase_transitions:
    ax.annotate(label, xy=(num, score), xytext=(num, score + 0.025),
               fontsize=8, ha='center', fontweight='bold',
               arrowprops=dict(arrowstyle='->', color='dimgray', lw=1),
               bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

# Horizontal best line
ax.axhline(y=0.2259, color=C['best'], ls='--', alpha=0.4, lw=1)
ax.text(340, 0.2259, '$S^* = 0.2259$', fontsize=9, color=C['best'], va='bottom')

ax.set_xlabel('Experiment Number', fontsize=12)
ax.set_ylabel('Overall Score (Phase 2)', fontsize=12)
ax.set_title('Figure 8: Complete Experiment Trajectory (338 experiments)',
            fontsize=14, fontweight='bold')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_xlim(-5, 345)
ax.set_ylim(0, 0.28)

plt.tight_layout()
plt.show()

---
## 10. 현재 최적 모델의 지표 분해

### exp_0338: 요스트 법칙 ($p=4.0$) + 시그모이드 바닥값 + 확산 활성화 ($eta=2.0$)

In [ ]:
# ============================================================
# Figure 9: Metric Decomposition — Radar + Waterfall
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- (a) Radar Chart ---
ax = axes[0]
ax.remove()
ax = fig.add_subplot(121, polar=True)

categories = ['Recall\n(0.415)', 'MRR\n(0.248)', 'Precision\nLift (0.0)', 
              'Correlation\n(0.302)', 'Smoothness\n(0.899)']
N = len(categories)

# Baseline values (normalized)
baseline_vals = [0.034, 0.024, 0.004, 0.690, 0.900]
best_vals =     [0.415, 0.248, 0.000, 0.302, 0.899]
# Theoretical max
max_vals =      [0.415, 0.400, 0.200, 1.000, 1.000]

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the polygon

baseline_vals += baseline_vals[:1]
best_vals += best_vals[:1]
max_vals += max_vals[:1]

ax.plot(angles, max_vals, 'o-', color='gray', lw=1, alpha=0.3, label='Theoretical Max')
ax.fill(angles, max_vals, color='gray', alpha=0.05)
ax.plot(angles, baseline_vals, 'o--', color=C['exp'], lw=1.5, alpha=0.7, label='Baseline')
ax.fill(angles, baseline_vals, color=C['exp'], alpha=0.1)
ax.plot(angles, best_vals, 'o-', color=C['jost'], lw=2.5, label='exp_0338 (Best)')
ax.fill(angles, best_vals, color=C['jost'], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 1.05)
ax.set_title('(a) Metric Radar', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='lower right', bbox_to_anchor=(1.3, -0.1), fontsize=9)

# --- (b) Score Waterfall ---
ax2 = axes[1]

components = ['Recall\n(0.40×0.415)', 'MRR\n(0.30×0.248)', 'Prec. Lift\n(0.30×0.0)',
             '= Retrieval', '×(0.85+0.15P)', '= Overall']
values = [0.40*0.415, 0.30*0.248, 0.30*0.0, 0.2403, None, 0.2259]

# Waterfall
bottoms = [0, 0.166, 0.166+0.0744, 0, 0, 0]
heights = [0.166, 0.0744, 0.0, 0.2403, 0.2403, 0.2259]
colors_bar = [C['exp'], C['quart'], C['quad'], C['jost'], C['baseline'], C['best']]

bars = ax2.bar(range(len(components)), heights, bottom=bottoms,
              color=colors_bar, alpha=0.85, edgecolor='white', linewidth=1.5)

# Value labels
for i, (h, b) in enumerate(zip(heights, bottoms)):
    if h > 0.01:
        ax2.text(i, b + h + 0.005, f'{h:.4f}', ha='center', fontsize=9, fontweight='bold')
    elif i == 2:
        ax2.text(i, b + 0.005, '0.0000', ha='center', fontsize=9, color='red')

ax2.set_xticks(range(len(components)))
ax2.set_xticklabels(components, fontsize=8.5)
ax2.set_ylabel('Score Contribution')
ax2.set_title('(b) Score Decomposition Waterfall', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 0.30)

# Annotate bottleneck
ax2.annotate('BOTTLENECK\n(precision_lift = 0)', xy=(2, 0.01), xytext=(2.5, 0.08),
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=2))

plt.suptitle('Figure 9: Current Best (exp_0338) — Metric Decomposition',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 11. 시그모이드 바닥값 역학

시그모이드 바닥값은 "보호되는" 기억과 "소모 가능한" 기억 사이에 매끄럽고 미분 가능한 경계를 형성한다. 이 시각화는 바닥값 함수가 중요도를 최소 활성화 수준으로 어떻게 매핑하는지 보여준다.

In [ ]:
# ============================================================
# Figure 10: Sigmoid Floor & Activation Phase Space
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- (a) Sigmoid Floor Function ---
ax = axes[0]

impact = np.linspace(0, 1, 200)
alpha, rho = 1.5, 1.0

# For different stability levels
for s, ls, label in [(0.0, '--', 'stability = 0.0'), 
                      (0.5, '-.', 'stability = 0.5'),
                      (1.0, '-', 'stability = 1.0')]:
    importance = (impact * alpha + s * rho) / (alpha + rho)
    z = 20.0 * (importance - 0.25)
    sigmoid = 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50)))
    floor = 0.60 * sigmoid
    ax.plot(impact, floor, ls=ls, lw=2.5, label=label)

ax.fill_between(impact, 0, 0.60 / (1 + np.exp(-20 * (impact * 1.5/2.5 - 0.25))),
               alpha=0.1, color=C['jost'])

# Annotate regions
ax.annotate('Expendable zone\n(rapid decay)', xy=(0.1, 0.05), fontsize=10,
           color=C['exp'], fontweight='bold',
           bbox=dict(facecolor='white', alpha=0.8, edgecolor=C['exp']))
ax.annotate('Protected zone\n(floor retention)', xy=(0.7, 0.45), fontsize=10,
           color=C['jost'], fontweight='bold',
           bbox=dict(facecolor='white', alpha=0.8, edgecolor=C['jost']))

ax.axhline(y=0.60, color='gray', ls=':', alpha=0.3)
ax.text(1.02, 0.60, '$f_{\\max}$', fontsize=10, color='gray')
ax.axvline(x=0.25 * 2.5/1.5, color='gray', ls=':', alpha=0.3)  # approx midpoint

ax.set_xlabel('Memory Impact $I$', fontsize=12)
ax.set_ylabel('Floor Activation $f$', fontsize=12)
ax.set_title(r'(a) Sigmoid Floor: $f = 0.60 \cdot \sigma(20(I^\prime - 0.25))$')
ax.legend(loc='upper left', framealpha=0.9)
ax.set_ylim(0, 0.7)

# --- (b) Activation Phase Space ---
ax2 = axes[1]

# Simulate trajectories for different impact levels
import math

def full_jost_decay(impact_val, stab=0.5, steps=200):
    a = 1.0
    trajectory = [a]
    alpha, rho = 1.5, 1.0
    importance = (impact_val * alpha + stab * rho) / (alpha + rho)
    z = 20.0 * (importance - 0.25)
    floor = 0.60 / (1 + math.exp(-z)) if z < 50 else 0.60
    
    lam_base = 0.020  # average of fact/episode
    retention = max(1.0 + alpha * impact_val + rho * stab, 1.0)
    lam_eff = lam_base / retention
    
    for _ in range(steps):
        excess = max(a - floor, 0)
        decay = lam_eff * excess**4.0
        a = max(a - decay, floor)
        trajectory.append(a)
    return np.array(trajectory), floor

t = np.arange(201)
for imp, color, alpha_val in [(0.1, '#e74c3c', 0.8), (0.3, '#e67e22', 0.8),
                               (0.5, '#2ecc71', 0.9), (0.7, '#3498db', 0.9),
                               (0.9, '#1abc9c', 0.9)]:
    traj, floor = full_jost_decay(imp)
    ax2.plot(t, traj, color=color, lw=2, alpha=alpha_val, label=f'impact={imp:.1f} (floor={floor:.2f})')
    ax2.axhline(y=floor, color=color, ls=':', alpha=0.2, lw=0.8)

ax2.set_xlabel('Tick', fontsize=12)
ax2.set_ylabel('Activation $a(t)$', fontsize=12)
ax2.set_title('(b) Activation Trajectories by Impact Level')
ax2.legend(loc='upper right', framealpha=0.9, fontsize=8.5)
ax2.set_ylim(0, 1.05)

plt.suptitle('Figure 10: Sigmoid Floor & Activation Dynamics',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 12. 구조적 병목 분석

### 12.1 재현율 상한

임베딩 모델 ()은 약 $pprox 0.415$의 **유사도 재현율** — 활성화 임계값에 관계없이 정답 기억이 상위 5개 결과에 나타나는 테스트 쿼리의 비율 — 을 달성한다. 이는 허용된 탐색 표면 내에서의 **변경 불가능한 상한**이다.

### 12.2 정밀도 향상 = 0

가장 치명적인 병목: 모든 요스트 실험에서 **정밀도 향상이 0**이다. 이는 감쇠 함수가 관련 기억을 임계값 위에 유지하면서 비관련 기억을 아래로 밀어내는 데 실패함을 의미한다. 수식으로:

$$\Delta_P = P_{	ext{strict}} - rac{\overline{	ext{Recall}}}{k} pprox 0.083 - 0.083 = 0$$

이는 **상위-$k$ 결과 내에서 활성화와 관련성이 상관되지 않음**을 시사한다 — 감쇠 함수가 특정 쿼리에 답하는 기억을 선택적으로 보존하지 못하고 있다.

### 12.3 이론적 상한

현재 제약 조건 하에서:

$$S^*_{	ext{upper}} = (0.40 	imes 0.415 + 0.30 	imes 0.25 + 0.30 	imes \Delta_P) 	imes (0.85 + 0.15 	imes S_P)$$

| 시나리오 | $\Delta_P$ | $S_P$ | $S^*$ |
|---------|-----------|-------|-------|
| 현재 (정밀도 향상 없음) | 0.00 | 0.60 | 0.226 |
| 중간 향상 | 0.05 | 0.65 | 0.255 |
| 강한 향상 | 0.10 | 0.70 | 0.285 |
| 이론적 최댓값 | 0.20 | 1.00 | 0.346 |

In [ ]:
# ============================================================
# Figure 11: Theoretical Upper Bound Sensitivity
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- (a) Overall Score vs Precision Lift ---
ax = axes[0]

precision_lifts = np.linspace(0, 0.20, 100)
plaus_scenarios = [0.50, 0.60, 0.70, 0.80, 1.00]

for sp, ls in zip(plaus_scenarios, ['-', '--', '-.', ':', '-']):
    retrieval = 0.40 * 0.415 + 0.30 * 0.248 + 0.30 * precision_lifts
    overall = retrieval * (0.85 + 0.15 * sp)
    ax.plot(precision_lifts, overall, ls=ls, lw=2, label=f'$S_P = {sp:.2f}$')

# Current position
ax.scatter([0.0], [0.2259], s=150, c=C['exp'], edgecolors='black', zorder=5)
ax.annotate('Current\n($S=0.226$)', xy=(0.0, 0.2259), xytext=(0.03, 0.20),
           fontsize=10, fontweight='bold', color=C['exp'],
           arrowprops=dict(arrowstyle='->', color=C['exp'], lw=1.5))

ax.set_xlabel(r'Precision Lift $\Delta_P$', fontsize=12)
ax.set_ylabel('Achievable Overall Score', fontsize=12)
ax.set_title('(a) Sensitivity to Precision Lift')
ax.legend(title='Plausibility', loc='upper left', framealpha=0.9)
ax.set_ylim(0.15, 0.40)

# --- (b) Contribution Gap Analysis ---
ax2 = axes[1]

components = ['Recall\nContribution', 'MRR\nContribution', 'Precision Lift\nContribution',
             'Plausibility\nMultiplier']
current = [0.40*0.415, 0.30*0.248, 0.30*0.0, 0.15*0.60]
potential = [0.40*0.415, 0.30*0.300, 0.30*0.10, 0.15*0.80]
gap = [p - c for c, p in zip(current, potential)]

x = np.arange(len(components))
width = 0.3

ax2.bar(x - width/2, current, width, label='Current', color=C['jost'], alpha=0.85)
ax2.bar(x + width/2, gap, width, bottom=current, label='Potential Gain', 
       color=C['accent'], alpha=0.7, hatch='//')

for i, (c, g) in enumerate(zip(current, gap)):
    ax2.text(i - width/2, c + 0.002, f'{c:.3f}', ha='center', fontsize=8)
    if g > 0.001:
        ax2.text(i + width/2, c + g + 0.002, f'+{g:.3f}', ha='center', fontsize=8, color=C['accent'])

ax2.set_xticks(x)
ax2.set_xticklabels(components, fontsize=9)
ax2.set_ylabel('Score Contribution')
ax2.set_title('(b) Current vs. Potential Contributions')
ax2.legend(framealpha=0.9)
ax2.set_ylim(0, 0.22)

plt.suptitle('Figure 11: Structural Bottleneck — Theoretical Upper Bound',
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 13. 가설 및 결과 요약

### 13.1 가설 계보

다음 표는 가설의 전체 계보, 수학적 혁신, 그리고 실험적 판정을 추적한다.

In [ ]:
# ============================================================
# Figure 12: Hypothesis Genealogy — Impact vs. Result
# ============================================================

fig, ax = plt.subplots(figsize=(14, 7))

# Key hypotheses with their impact
hypotheses = [
    ('Power-law tail\n(quadratic→cubic)', 'Phase I', 0.37, True, 
     r'$a(1-\lambda a^2/c)$: $1/\sqrt{t}$ tail'),
    ('Exponential impact\nprotection', 'Phase I', 0.02, True,
     r'$\exp(\alpha I)$ vs $(1+\alpha I)$'),
    ('Episode:Fact\nratio=3.0', 'Phase I', 0.003, True,
     r'Pareto optimal $r^*=3.04$'),
    ('Hard floor\n$\\beta_0=0.79$', 'Phase II', 0.03, True,
     'Maximum recall retention'),
    ('Sqrt-floor\ndecay', 'Phase II', 0.06, True,
     r'$f = \sqrt{I} \cdot \gamma$'),
    ('Scoring reform +\nactivation ranking', 'Pivot', 0.0, True,
     'New optimization landscape'),
    ("Jost's Law\n$p=4.0$", 'Phase III', 0.20, True,
     r'$\dot{a} = -\lambda(a-f)^4$'),
    ('Activation\nweight $w=0.5$', 'Phase III', 0.0, True,
     r'$\text{score} = \cos \cdot a^{0.5}$'),
    ('Spreading\nactivation', 'Phase III', 0.003, True,
     r'$\times(1+\beta\bar{a}_{\mathcal{N}})$'),
    ('Equilibrium\nconvergence', 'Phase I', -0.02, False,
     'Convergence too slow'),
    ('Memory\nconsolidation', 'Phase I', -0.01, False,
     'All pass threshold → corr=0'),
    ('Bi-exponential', 'Phase III', -0.03, False,
     'Slow component too strong'),
    ('Hyperbolic +\nacceleration', 'Phase III', -0.08, False,
     r'Jost excess$^4$ superior'),
    ('$p=5.0$', 'Phase III', -0.01, False,
     'Over-aggressive decay'),
]

# Sort by improvement
hypotheses.sort(key=lambda x: x[2], reverse=True)

y_pos = range(len(hypotheses))
colors_h = [C['jost'] if h[3] else C['exp'] for h in hypotheses]
improvements = [h[2] for h in hypotheses]

bars = ax.barh(y_pos, improvements, color=colors_h, alpha=0.8, edgecolor='white', linewidth=1)

# Labels
ax.set_yticks(y_pos)
ax.set_yticklabels([h[0] for h in hypotheses], fontsize=9)

# Phase badges
for i, h in enumerate(hypotheses):
    # Math annotation on the right
    x_pos = max(h[2] + 0.01, 0.01) if h[2] >= 0 else h[2] - 0.01
    ha = 'left' if h[2] >= 0 else 'right'
    ax.text(x_pos, i, h[4], fontsize=7.5, va='center', ha=ha,
           color='dimgray', style='italic')

ax.axvline(x=0, color='black', lw=1)
ax.set_xlabel(r'Score Improvement ($\Delta S$)', fontsize=12)
ax.set_title('Figure 12: Hypothesis Genealogy — Confirmed vs. Rejected',
            fontsize=14, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=C['jost'], alpha=0.8, label='Confirmed'),
                   Patch(facecolor=C['exp'], alpha=0.8, label='Rejected')]
ax.legend(handles=legend_elements, loc='lower right', framealpha=0.9)

ax.invert_yaxis()
ax.set_xlim(-0.12, 0.45)
plt.tight_layout()
plt.show()

---
## 14. 최종 모델: 완전한 명세

### 14.1 감쇠 함수

$$oxed{a_{t+1} = \max\left(a_t - rac{\lambda_{	ext{base}}}{1 + lpha I + ho s} \cdot \left(\max(a_t - f, 0)ight)^4, \;\; f ight)}$$

여기서:

$$f = 0.60 \cdot \sigma\!\left(20 \cdot \left(rac{1.5 I + s}{2.5} - 0.25ight)ight), \quad \lambda_{	ext{base}} = egin{cases} 0.008 & 	ext{사실(fact)인 경우} \\ 0.035 & 	ext{에피소드(episode)인 경우} \end{cases}$$

### 14.2 검색 순위

$$	ext{score}(q, m) = 	ext{sim}_{\cos}(q, m) \cdot a_m^{0.5} \cdot \left(1 + 2.0 \cdot \overline{a}_{\mathcal{N}(m)}ight)$$

### 14.3 최적 매개변수

| 매개변수 | 기호 | 값 | 역할 |
|---------|------|-----|------|
|  | $\lambda_f$ | 0.008 | 기본 사실 감쇠율 |
|  | $\lambda_e$ | 0.035 | 기본 에피소드 감쇠율 (비율 4.375:1) |
|  | $lpha$ | 1.5 | 중요도 보호 강도 |
|  | $ho$ | 1.0 | 안정성 영향도 |
|  | $f_{\max}$ | 0.60 | 최대 바닥값 수준 |
|  | $k$ | 20.0 | 시그모이드 급경사도 |
|  | $\mu$ | 0.25 | 시그모이드 중간점 |
|  | $p$ | 4.0 | 초과분 감쇠 지수 |
|  | $w$ | 0.5 | 순위 활성화 영향도 |
|  | $eta$ | 2.0 | 확산 활성화 강도 |

---
## 15. 핵심 발견 및 교훈

### 15.1 수학적 구조 $\gg$ 매개변수 조정

가장 큰 단일 향상 (지수 $ightarrow$ 멱법칙: $+0.37$)은 매개변수 조정이 아닌 **수학적 형태의 변경**에서 비롯되었다. 338회의 실험에 걸쳐, 구조적 변경 (곡선 형태, 바닥값 구조, 요스트 법칙)이 전체 향상의 약 95%를 기여하였다.

### 15.2 평가 설계가 탐색 궤적을 결정한다

1단계의 재현율 중심 점수 체계는 296회의 실험을 지역 최솟값("바닥값 편법")으로 몰아넣었다. 2단계 점수 개편은 즉시 새로운 최적화 경관을 열었다. **평가 함수가 어떤 프롬프트나 지시보다 에이전트 행동을 더 강하게 형성한다.**

### 15.3 수렴은 신호이지 중단 기준이 아니다

세 번의 수렴 사건이 발생하였으며 (exp_0019, exp_0296, exp_0332), 각각 더 많은 매개변수 탐색이 아닌 **구조적 혁신**으로 해결되었다:
1. 삼차 감쇠 정체 $ightarrow$ 바닥값 구조
2. 바닥값 편법 정체 $ightarrow$ 점수 개편 + 활성화 가중 순위
3. 활성화 가중치 정체 $ightarrow$ 확산 활성화

### 15.4 미해결 문제

1. **정밀도 향상 = 0**: 감쇠 함수가 쿼리 관련 기억을 선택적으로 보존하지 못한다. 이를 위해서는 쿼리 의존적 감쇠 또는 허용된 표면 외부의 구조적 변경이 필요하다.
2. **재현율 상한 41.5%**: 감쇠가 아닌 임베딩 품질()에 의해 제한됨.
3. **수확 체감**: 마지막 38회 실험 (exp_0300–0338)은 총 $+0.003$의 향상만을 가져왔으며, 현재 탐색 표면이 거의 소진에 가까울 수 있음을 시사한다.

---

*338회 실험 $\cdot$ 10개 구조적 에포크 $\cdot$ 기준선 $0.021 ightarrow$ 최적 $0.226$ $(10.7	imes)$*

*보고서 생성일: 2026-03-19*